# XGBoost benchmark — v5 per-security, NOK swap rates (fleet notebook)

**Matched-pair anchor.** Identical to `model_htboost_v5_clean.ipynb` in target, features
(via the shared `src.features.*` modules — verified byte-equivalent to the frozen notebook's
inline `_v4` functions), PCA compression (the *same* `src.features.pca.fit_transform_xm_pca`),
walk-forward folds + one-sided `H-1` purge, block-CV/LORO two-sided purge+embargo, seed
(`config.JULIA_SEED`, `n_jobs=1`), and evaluation harness (`SHARED_COLS` / `compute_metrics_row`).
**Do not change any of those** — they make the §2.5 HTBoost-vs-XGBoost DM-H valid.

**Documented differences (thesis):** squared-error loss (`reg:squarederror`) vs HTBoost's
Student-t; XGBoost tunes lr/depth/n_estimators(early-stop)/subsample/colsample/reg_lambda/
min_child_weight on a time-ordered inner split (training-window only).

**Extraction (per the no-re-run rule):** per-obs predictions (DM-H), per-fold metrics
(SHARED_COLS), full per-fold tuning log (§5.3.1 appendix), **all** importance types
(gain/weight/cover/total_gain/total_cover), **SHAP** (values + swarm), **learning curves**
(`evals_result`), plus the **block-CV/LORO learnability family**. All machine-stamped,
auto-pushed per horizon, and consolidatable.

## Setup

In [ ]:
import argparse, hashlib, itertools, json, os, pickle, re, sys, time, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

_ROOT = os.getcwd()
while _ROOT != os.path.dirname(_ROOT) and not os.path.isdir(os.path.join(_ROOT, 'src')):
    _ROOT = os.path.dirname(_ROOT)
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

from src.data.bloomberg import load_data
from src.data.norway import load_norway_raw, print_connectivity_report
from src.features.macro import add_macro_features
from src.features.norway import add_norway_features
from src.features.panel import build_panel
from src.features.pca import fit_transform_xm_pca
from src.features.taxonomy import bucket_feature, BUCKETS
from src.evaluation.metrics import (compute_metrics_row, SHARED_COLS, MTC_COLS,
                                    apply_mtc_family, finalize_long_csv)
import src.config as config
from src.config import MACHINE_ID, WF_FOLDS, FOLD_NAMES, BLOCK_CV_FOLDS, EMBARGO_FOR_H
from src.utils.gitpush import push_results
try:
    from thesis_style import apply_thesis_style, OKABE_ITO
    apply_thesis_style()
except Exception:
    OKABE_ITO = ['#0072B2','#E69F00','#009E73','#D55E00','#CC79A7','#56B4E9','#F0E442','#000000']
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')
# SHAP is optional: VALUES come from native pred_contribs (no dep); only the swarm aesthetic uses shap.
try:
    import shap; _HAVE_SHAP = True
except Exception:
    _HAVE_SHAP = False
print(f'Imports OK (xgboost {xgb.__version__}; shap available={_HAVE_SHAP})')

### Configuration

In [ ]:
# ── CONFIG — matched-pair core mirrors HTBoost; nothing selected on OOS performance ──
NOTEBOOK   = 'xgboost_v5'
MODEL_KIND = 'per_security'
IS_POOLED  = False
RUN_TS     = datetime.now(tz=timezone.utc).isoformat()

H_GRID      = config.H_GRID
H_GRID_LONG = [126, 252]
UNIVERSE_MIN_OBS = 500
MIN_TRAIN_OBS = config.MIN_TRAIN_OBS
MIN_TEST_OBS  = config.MIN_TEST_OBS
TARGET_LAGS   = 5
XM_PCA_VAR    = config.XM_PCA_VAR
XM_PCA_KMAX   = config.XM_PCA_KMAX
FEAT_SPEC     = f'pca_var{XM_PCA_VAR}_kmax{XM_PCA_KMAX}_tl{TARGET_LAGS}'
JULIA_SEED    = config.JULIA_SEED
META_COLS     = {'date', 'security', 'y', 'level'}

OUT_DIR   = os.path.join(_ROOT, 'results', 'xgb_v5_nor')
SMOKE_DIR = os.path.join(OUT_DIR, '_smoke')
FIG_DIR   = os.path.join(OUT_DIR, 'figures')
NOR_CACHE = os.path.join(_ROOT, 'data', 'cache', 'norway_raw_features.csv')
for _d in (OUT_DIR, FIG_DIR): os.makedirs(_d, exist_ok=True)
SMOKE_SEC, SMOKE_H = 'NOR_10Y', 21

# Documented difference: squared-error loss + a pre-committed grid tuned in-window only.
OBJECTIVE      = 'reg:squarederror'
N_TREES_MAX    = 500
EARLY_STOP_RND = 20
INNER_VAL_FRAC = 0.20
XGB_PARAM_GRID = {
    'learning_rate':    [0.01, 0.05, 0.10],
    'max_depth':        [3, 5],
    'subsample':        [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'reg_lambda':       [1.0, 5.0, 10.0],
    'min_child_weight': [1, 5],
}
IMP_TYPES = ('gain', 'weight', 'cover', 'total_gain', 'total_cover')

# Run switches
RUN_WF       = True
RUN_BLOCKCV  = True
RUN_LORO     = True
RUN_PUSH     = True     # auto-push per horizon via src.utils.gitpush (HTBoost parity)

def _p(out_dir, name): return os.path.join(out_dir, name)
def _wf_csv(h, d):        return _p(d, f'xgb_wf_H{h}__{MACHINE_ID}.csv')
def _wf_pred_csv(h, d):   return _p(d, f'xgb_wf_preds_H{h}__{MACHINE_ID}.csv')
def _wf_imp_csv(h, d):    return _p(d, f'xgb_wf_imps_H{h}__{MACHINE_ID}.csv')
def _blk_csv(h, d):       return _p(d, f'xgb_blockcv_H{h}__{MACHINE_ID}.csv')
def _blk_pred_csv(h, d):  return _p(d, f'xgb_blockcv_preds_H{h}__{MACHINE_ID}.csv')
def _blk_imp_csv(h, d):   return _p(d, f'xgb_blockcv_imps_H{h}__{MACHINE_ID}.csv')
def _tuning_csv(h, d):    return _p(d, f'xgb_tuning_log_H{h}__{MACHINE_ID}.csv')
def _evalhist_csv(h, d):  return _p(d, f'xgb_eval_history_H{h}__{MACHINE_ID}.csv')
def _shap_csv(h, d):      return _p(d, f'xgb_shap_buckets_H{h}__{MACHINE_ID}.csv')

def _maybe_push(paths, msg):
    paths = [p for p in paths if p and os.path.exists(p)]
    if RUN_PUSH and paths:
        try: push_results(paths, msg)
        except Exception as e: print(f'  [push] skipped — {repr(e)[:80]}')

print(f'OUT_DIR={OUT_DIR}')
print(f'WF folds={len(WF_FOLDS)}  block-CV folds={len(BLOCK_CV_FOLDS)}  seed={JULIA_SEED}')
print(f'grid={len(list(itertools.product(*XGB_PARAM_GRID.values())))} combos/fold  RUN_PUSH={RUN_PUSH}')

### Data load + universe

In [ ]:
def _config_hash(cfg, extra=''):
    payload = json.dumps({k: cfg.get(k) for k in sorted(cfg)}, sort_keys=True, default=str) + '|' + str(extra)
    return hashlib.md5(payload.encode()).hexdigest()[:12]

def _score(y, yhat):
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    m = np.isfinite(y) & np.isfinite(yhat)
    if m.sum() < 5: return None
    return {'dir_acc': float(np.mean(np.sign(y[m]) == np.sign(yhat[m]))),
            'r2': float(r2_score(y[m], yhat[m])), 'n_obs': int(m.sum())}

def _prepare_x_xgb(df):
    df = df.copy()
    df = df.rename(columns={c: re.sub(r'[^a-zA-Z0-9_]', '_', str(c)).strip('_') for c in df.columns})
    seen, cols = {}, []
    for c in df.columns:
        if c in seen: seen[c] += 1; cols.append(f'{c}_{seen[c]}')
        else: seen[c] = 0; cols.append(c)
    df.columns = cols
    return df.replace([np.inf, -np.inf], np.nan).astype(np.float64)

def _done_secs(csv_path):
    if not os.path.exists(csv_path): return set()
    try: return set(pd.read_csv(csv_path, usecols=['security'])['security'].astype(str))
    except Exception: return set()

def _append_csv(df, path):
    write_header = not os.path.exists(path)
    with open(path, 'a', newline='') as fh:
        df.to_csv(fh, header=write_header, index=False); fh.flush(); os.fsync(fh.fileno())

def load_and_augment_data():
    print('Loading Bloomberg data ...')
    df_raw = load_data()
    a, b = df_raw.index.min().strftime('%Y-%m-%d'), df_raw.index.max().strftime('%Y-%m-%d')
    nor_raw, nor_report = load_norway_raw(a, b, NOR_CACHE, live=False)
    print_connectivity_report(nor_report)
    if not nor_raw.empty:
        df_raw = df_raw.join(nor_raw, how='left')
        df_raw = add_norway_features(df_raw, nor_raw)
    df_raw = add_macro_features(df_raw)
    print(f'  df_raw after augmentation: {df_raw.shape}')
    return df_raw

df_raw = load_and_augment_data()
_SWAP_PAT = re.compile(r'^[A-Z]+_\d+[WMY]$')
swap_cols = sorted(c for c in df_raw.columns if _SWAP_PAT.match(c))
obs = df_raw[swap_cols].notna().sum()
UNIVERSE = [s for s in sorted(obs[obs >= UNIVERSE_MIN_OBS].index.tolist()) if s.rsplit('_', 1)[0] == 'NOR']
print(f'Norwegian universe ({len(UNIVERSE)}): {UNIVERSE}')
assert SMOKE_SEC in UNIVERSE

### Feature taxonomy (shared `src.features.taxonomy`)

In [ ]:
_tax_panel = build_panel(df_raw, [SMOKE_SEC], SMOKE_H)
_tax_feats = [c for c in _tax_panel.columns if c not in META_COLS]
feature_taxonomy = pd.DataFrame({'feature': _tax_feats, 'bucket': [bucket_feature(c) for c in _tax_feats]})
feature_taxonomy.to_csv(_p(OUT_DIR, 'xgb_feature_taxonomy.csv'), index=False)
print(feature_taxonomy['bucket'].value_counts().to_string())
print('NOTE: xmpca_* (post-PCA) bucket to cross_market — same as HTBoost/linear.')

### XGBoost tune + fit + extraction (importances · SHAP · learning curve)

In [ ]:
def _tune_and_fit_xgb(y_tr, x_tr_df, y_te, x_te_df, H, seed=JULIA_SEED):
    """Time-ordered inner-CV grid search + refit. CORE fold/tuning logic is verbatim from the
    verified script; adds extraction of all importance types, SHAP, and the learning curve."""
    n = len(y_tr)
    n_inner_val = max(MIN_TEST_OBS, int(n * INNER_VAL_FRAC))
    n_inner_tr  = n - n_inner_val
    n_purge     = max(0, H - 1)                 # inner purge mirrors the outer H-1 purge
    n_it_purged = n_inner_tr - n_purge
    if n_it_purged >= MIN_TRAIN_OBS:
        X_it, y_it, use_es = x_tr_df.iloc[:n_it_purged], y_tr[:n_it_purged], True
    else:
        X_it, y_it, use_es = x_tr_df.iloc[:n_inner_tr], y_tr[:n_inner_tr], False
    X_iv, y_iv = x_tr_df.iloc[n_inner_tr:], y_tr[n_inner_tr:]

    keys, values = list(XGB_PARAM_GRID.keys()), list(XGB_PARAM_GRID.values())
    best_mse, best_params, best_n_est, tuning_rows = np.inf, None, N_TREES_MAX, []
    for combo in itertools.product(*values):
        params = dict(zip(keys, combo))
        if use_es:
            mdl = xgb.XGBRegressor(n_estimators=N_TREES_MAX, objective=OBJECTIVE,
                                   early_stopping_rounds=EARLY_STOP_RND, random_state=seed,
                                   n_jobs=1, verbosity=0, **params)
            mdl.fit(X_it, y_it, eval_set=[(X_iv, y_iv)], verbose=False)
            n_est = int(getattr(mdl, 'best_iteration', N_TREES_MAX - 1)) + 1
        else:
            mdl = xgb.XGBRegressor(n_estimators=N_TREES_MAX, objective=OBJECTIVE,
                                   random_state=seed, n_jobs=1, verbosity=0, **params)
            mdl.fit(X_it, y_it); n_est = N_TREES_MAX
        mse_iv = float(np.mean((y_iv - mdl.predict(X_iv)) ** 2))
        tuning_rows.append({**params, 'val_mse': mse_iv, 'best_n_estimators': n_est, 'selected': False})
        if mse_iv < best_mse:
            best_mse, best_params, best_n_est = mse_iv, params.copy(), n_est
    for r in tuning_rows:
        if all(r[k] == best_params[k] for k in keys): r['selected'] = True; break

    # Final model on the FULL training window (predictions)
    mdl_final = xgb.XGBRegressor(n_estimators=best_n_est, objective=OBJECTIVE,
                                 random_state=seed, n_jobs=1, verbosity=0, **best_params)
    mdl_final.fit(x_tr_df, y_tr)
    yhat_tr, yhat_te = mdl_final.predict(x_tr_df), mdl_final.predict(x_te_df)
    booster = mdl_final.get_booster()

    # (#4) ALL importance types, 0-filled for unused features
    feats = list(x_tr_df.columns)
    imp_by_type = {t: {fn: float(booster.get_score(importance_type=t).get(fn, 0.0)) for fn in feats}
                   for t in IMP_TYPES}

    # (#5) SHAP via native pred_contribs (no shap dep needed for values); mean|SHAP| per feature on OOS
    shap_mean_abs = {}
    try:
        contribs = booster.predict(xgb.DMatrix(x_te_df, feature_names=feats), pred_contribs=True)
        sv = np.abs(contribs[:, :-1]).mean(axis=0)     # last column is the bias term
        shap_mean_abs = {fn: float(v) for fn, v in zip(feats, sv)}
    except Exception as e:
        print(f'    [shap] skipped — {repr(e)[:60]}')

    # (#6) learning curve for the SELECTED config (one extra fit; train+val RMSE per round)
    eval_history = []
    try:
        mc = xgb.XGBRegressor(n_estimators=best_n_est, objective=OBJECTIVE, random_state=seed,
                              n_jobs=1, verbosity=0, eval_metric='rmse', **best_params)
        mc.fit(X_it, y_it, eval_set=[(X_it, y_it), (X_iv, y_iv)], verbose=False)
        ev = mc.evals_result()
        tr_c, va_c = ev['validation_0']['rmse'], ev['validation_1']['rmse']
        eval_history = [{'round': i, 'train_rmse': float(t), 'val_rmse': float(v)}
                        for i, (t, v) in enumerate(zip(tr_c, va_c))]
    except Exception as e:
        print(f'    [eval_history] skipped — {repr(e)[:60]}')

    return dict(yhat_tr=yhat_tr, yhat_te=yhat_te, best_params=best_params, best_n_est=best_n_est,
                tuning_rows=tuning_rows, imp_by_type=imp_by_type, shap_mean_abs=shap_mean_abs,
                eval_history=eval_history)
print('_tune_and_fit_xgb defined (core verbatim; +all-imp +shap +eval_history)')

### Shared fold scorer + walk-forward / block-CV·LORO runners

In [ ]:
def _fit_score_fold(tr, te, fc, H, sec, country, tenor, scheme, fold_name, regime, seed=JULIA_SEED):
    """PCA-compress (train only) -> tune+fit -> SHARED_COLS train+oos rows + preds + imps + tuning + curve.
    Identical fit path for WF and block-CV/LORO; only the train/test masks and `scheme` differ."""
    x_tr, x_te, pca = fit_transform_xm_pca(tr[fc], te[fc])
    fcount = x_tr.shape[1]
    x_tr_x, x_te_x = _prepare_x_xgb(x_tr), _prepare_x_xgb(x_te)
    y_tr, y_te = tr['y'].to_numpy(float), te['y'].to_numpy(float)
    r = _tune_and_fit_xgb(y_tr, x_tr_x, y_te, x_te_x, H, seed=seed)
    cfg = {**r['best_params'], 'n_estimators': r['best_n_est'], 'objective': OBJECTIVE,
           'inner_val_frac': INNER_VAL_FRAC}
    meta = dict(notebook=NOTEBOOK, run_ts=RUN_TS, model_kind=MODEL_KIND, is_pooled=IS_POOLED,
                validation_scheme=scheme, target_kind='level_change', security=sec, country=country,
                tenor=tenor, fold=fold_name, regime=regime,
                config_hash=_config_hash(cfg, extra=FEAT_SPEC), feature_count=fcount)
    rows = []
    for samp, yt, yh in (('train', y_tr, r['yhat_tr']), ('oos', y_te, r['yhat_te'])):
        row = compute_metrics_row(yt, yh, H, {**meta, 'sample': samp})
        row['pca_k'] = pca['k']; row['xm_pca_evr'] = pca['evr_cum_k']
        row['xgb_n_trees'] = r['best_n_est']
        row['xgb_max_depth'] = r['best_params'].get('max_depth', -1)
        row['xgb_lr'] = r['best_params'].get('learning_rate', -1)
        rows.append(row)
    preds = pd.DataFrame({'date': te['date'].values, 'security': sec, 'tenor': tenor,
                          'horizon': int(H), 'regime': regime, 'scheme': scheme,
                          'y_true': y_te, 'y_pred': np.asarray(r['yhat_te'], float)})
    imp_rec = {'security': sec, 'horizon': H, 'fold': fold_name, 'regime': regime, 'scheme': scheme,
               'imp_by_type': r['imp_by_type'], 'shap': r['shap_mean_abs']}
    for tr_row in r['tuning_rows']:
        tr_row.update({'security': sec, 'horizon': H, 'fold': fold_name, 'regime': regime, 'scheme': scheme})
    evh = [{'security': sec, 'horizon': H, 'fold': fold_name, 'scheme': scheme, **e} for e in r['eval_history']]
    return rows, imp_rec, preds, r['tuning_rows'], evh

def _run_security_wf(sec, df_raw, H, seed=JULIA_SEED, verbose=False):
    if sec not in df_raw.columns: return [], [], [], [], []
    panel = build_panel(df_raw, [sec], H)
    if len(panel) == 0: return [], [], [], [], []
    fc = [c for c in panel.columns if c not in META_COLS]
    country, tenor = sec.rsplit('_', 1)
    rows, imps, preds, tun, evh = [], [], [], [], []
    for fold_name, ts, te_end, regime in WF_FOLDS:
        ts_ts, te_ts = pd.Timestamp(ts), pd.Timestamp(te_end)
        purge_ts = ts_ts - pd.tseries.offsets.BDay(H - 1)            # one-sided H-1 purge
        trn = panel[panel['date'] < purge_ts].copy()
        tst = panel[(panel['date'] >= ts_ts) & (panel['date'] <= te_ts)].copy()
        if len(trn) < MIN_TRAIN_OBS or len(tst) < MIN_TEST_OBS: continue
        r, ir, pr, tr_rows, eh = _fit_score_fold(trn, tst, fc, H, sec, country, tenor,
                                                 'walk_forward', fold_name, regime, seed)
        rows += r; imps.append(ir); preds.append(pr); tun += tr_rows; evh += eh
    return rows, imps, preds, tun, evh

def _blockcv_entries(scheme):
    if scheme == 'block_cv':
        return [(name, regime, [(pd.Timestamp(s), pd.Timestamp(e))]) for (name, s, e, regime) in BLOCK_CV_FOLDS]
    if scheme == 'loro':
        out = []
        for regime in dict.fromkeys(r for (_n, _s, _e, r) in BLOCK_CV_FOLDS):
            segs = [(pd.Timestamp(s), pd.Timestamp(e)) for (_n, s, e, r) in BLOCK_CV_FOLDS if r == regime]
            out.append((f'LORO_{regime}', regime, segs))
        return out
    raise ValueError(scheme)

def _run_security_blockcv(sec, df_raw, scheme, H, seed=JULIA_SEED):
    """Non-causal learnability diagnostic: two-sided purge+embargo around each held-out block."""
    if sec not in df_raw.columns: return [], [], [], [], []
    panel = build_panel(df_raw, [sec], H)
    if len(panel) == 0: return [], [], [], [], []
    fc = [c for c in panel.columns if c not in META_COLS]
    country, tenor = sec.rsplit('_', 1)
    gap = (H - 1) + EMBARGO_FOR_H(H)
    dates = panel['date']
    rows, imps, preds, tun, evh = [], [], [], [], []
    for label, regime, segs in _blockcv_entries(scheme):
        test_mask = pd.Series(False, index=panel.index)
        for (s, e) in segs: test_mask |= (dates >= s) & (dates <= e)
        train_mask = ~test_mask
        for (s, e) in segs:                                         # two-sided symmetric gap
            lo, hi = s - pd.tseries.offsets.BDay(gap), e + pd.tseries.offsets.BDay(gap)
            train_mask &= ~((dates >= lo) & (dates <= hi))
        trn, tst = panel[train_mask].copy(), panel[test_mask].copy()
        if len(trn) < MIN_TRAIN_OBS or len(tst) < MIN_TEST_OBS: continue
        r, ir, pr, tr_rows, eh = _fit_score_fold(trn, tst, fc, H, sec, country, tenor,
                                                 scheme, label, regime, seed)
        rows += r; imps.append(ir); preds.append(pr); tun += tr_rows; evh += eh
    return rows, imps, preds, tun, evh

def _imps_long(imp_recs):
    """Flatten importance records: one row per (fold, feature, importance_type) + a shap_mean_abs type."""
    out = []
    for r in imp_recs:
        for t, d in r['imp_by_type'].items():
            for feat, val in d.items():
                out.append({'security': r['security'], 'horizon': r['horizon'], 'fold': r['fold'],
                            'regime': r['regime'], 'scheme': r['scheme'], 'feature': feat,
                            'importance_type': t, 'value': val, 'bucket': bucket_feature(feat)})
        for feat, val in (r.get('shap') or {}).items():
            out.append({'security': r['security'], 'horizon': r['horizon'], 'fold': r['fold'],
                        'regime': r['regime'], 'scheme': r['scheme'], 'feature': feat,
                        'importance_type': 'shap_mean_abs', 'value': val, 'bucket': bucket_feature(feat)})
    return pd.DataFrame(out)

def _shap_buckets(imp_recs):
    """Per-fold SHAP mass aggregated to taxonomy buckets (the cross-model signed-importance axis)."""
    rows = []
    for r in imp_recs:
        agg = {b: 0.0 for b in BUCKETS}
        for feat, val in (r.get('shap') or {}).items():
            agg[bucket_feature(feat)] += abs(float(val))
        tot = sum(agg.values()) or 1.0
        for b in BUCKETS:
            rows.append({'security': r['security'], 'horizon': r['horizon'], 'fold': r['fold'],
                         'scheme': r['scheme'], 'bucket': b, 'shap_share_pct': 100.0 * agg[b] / tot})
    return pd.DataFrame(rows)
print('runners defined: WF + block_cv/loro (shared _fit_score_fold); _imps_long, _shap_buckets')

### Smoke test (NOR_10Y × H=21, all WF folds)

In [ ]:
os.makedirs(SMOKE_DIR, exist_ok=True)
rows, imps, preds, tun, evh = _run_security_wf(SMOKE_SEC, df_raw, SMOKE_H, verbose=False)
assert rows, 'smoke produced no rows'
df_s = pd.DataFrame(rows); oos = df_s[df_s['sample'] == 'oos']
print(f'[SMOKE] {SMOKE_SEC} H={SMOKE_H}: {len(oos)} OOS folds')
for _, r in oos.iterrows():
    print(f'  {r["fold"]:12s} n={r["n_obs"]:4.0f} DA={r["dir_acc"]:.3f} CT-R2={r["ct_r2_oos"]:+.4f} '
          f'trees={r["xgb_n_trees"]:.0f} depth={r["xgb_max_depth"]:.0f}')
pd.DataFrame(rows).to_csv(_wf_csv(SMOKE_H, SMOKE_DIR), index=False)
pd.concat(preds, ignore_index=True).to_csv(_wf_pred_csv(SMOKE_H, SMOKE_DIR), index=False)
_imps_long(imps).to_csv(_wf_imp_csv(SMOKE_H, SMOKE_DIR), index=False)
pd.DataFrame(tun).to_csv(_tuning_csv(SMOKE_H, SMOKE_DIR), index=False)
pd.DataFrame(evh).to_csv(_evalhist_csv(SMOKE_H, SMOKE_DIR), index=False)
_shap_buckets(imps).to_csv(_shap_csv(SMOKE_H, SMOKE_DIR), index=False)
print('smoke outputs (incl. all-imp/shap/eval_history) written to', SMOKE_DIR)

### Horizon support gate

In [ ]:
def _horizon_supported(df_raw, h):
    panel = build_panel(df_raw, [SMOKE_SEC], h)
    if len(panel) == 0: return False
    for _, ts, te, _ in WF_FOLDS:
        ts_ts, te_ts = pd.Timestamp(ts), pd.Timestamp(te)
        purge = ts_ts - pd.tseries.offsets.BDay(h - 1)
        if (panel['date'] < purge).sum() >= MIN_TRAIN_OBS and \
           ((panel['date'] >= ts_ts) & (panel['date'] <= te_ts)).sum() >= MIN_TEST_OBS:
            return True
    return False

HORIZONS = list(H_GRID) + [h for h in H_GRID_LONG if _horizon_supported(df_raw, h)]
HORIZONS = sorted(set(HORIZONS))
print('Final horizon grid:', HORIZONS)

### Walk-forward sweep (per-horizon checkpoint + auto-push)

In [ ]:
def _load_pkl(p):
    if os.path.exists(p):
        try:
            with open(p, 'rb') as f: return pickle.load(f)
        except Exception: pass
    return []

def _dump_pkl(p, obj):
    with open(p, 'wb') as f: pickle.dump(obj, f); f.flush(); os.fsync(f.fileno())

if RUN_WF:
    for H_run in HORIZONS:
        wf_csv_p  = _wf_csv(H_run, OUT_DIR)
        imp_pkl_p = wf_csv_p.replace('.csv', '_imps.pkl')
        done      = _done_secs(wf_csv_p)
        all_imps  = _load_pkl(imp_pkl_p)            # resume-safe: reload accumulated importances
        if done: print(f'  [H={H_run}] resume: {len(done)} security(ies) already on disk')
        for sec in UNIVERSE:
            if sec in done: continue
            t0 = time.time()
            try:
                rows, imps, preds, tun, evh = _run_security_wf(sec, df_raw, H_run)
            except Exception as e:
                print(f'  [H={H_run}] {sec}: FAILED {repr(e)[:70]}'); continue
            if rows:  _append_csv(pd.DataFrame(rows), wf_csv_p)
            if preds: _append_csv(pd.concat(preds, ignore_index=True), _wf_pred_csv(H_run, OUT_DIR))
            if tun:   _append_csv(pd.DataFrame(tun), _tuning_csv(H_run, OUT_DIR))
            if evh:   _append_csv(pd.DataFrame(evh), _evalhist_csv(H_run, OUT_DIR))
            all_imps += imps
            _dump_pkl(imp_pkl_p, all_imps)
            _imps_long(all_imps).to_csv(_wf_imp_csv(H_run, OUT_DIR), index=False)     # full rewrite from accum
            _shap_buckets(all_imps).to_csv(_shap_csv(H_run, OUT_DIR), index=False)
            oos = [r for r in rows if r.get('sample') == 'oos']
            print(f'  [H={H_run}] {sec:<10s} folds={len(oos)} ({time.time()-t0:.1f}s)')
        _maybe_push([_wf_csv(H_run, OUT_DIR), _wf_pred_csv(H_run, OUT_DIR), _wf_imp_csv(H_run, OUT_DIR),
                     _tuning_csv(H_run, OUT_DIR), _evalhist_csv(H_run, OUT_DIR), _shap_csv(H_run, OUT_DIR)],
                    f'results: xgb_v5_nor walk_forward H{H_run} @ {MACHINE_ID}')
    print('WF sweep done')
else:
    print('[skipped] RUN_WF=False')

### Block-CV / LORO sweep (non-causal learnability family + auto-push)

In [ ]:
_schemes = (['block_cv'] if RUN_BLOCKCV else []) + (['loro'] if RUN_LORO else [])
if _schemes:
    for H_run in HORIZONS:
        blk_csv_p = _blk_csv(H_run, OUT_DIR)
        imp_pkl_p = blk_csv_p.replace('.csv', '_imps.pkl')
        done_keys = set()
        if os.path.exists(blk_csv_p):
            _d = pd.read_csv(blk_csv_p, usecols=['security', 'validation_scheme'])
            done_keys = set(_d['security'] + '|' + _d['validation_scheme'])
        blk_imps = _load_pkl(imp_pkl_p)
        for scheme in _schemes:
            for sec in UNIVERSE:
                if f'{sec}|{scheme}' in done_keys: continue
                t0 = time.time()
                try:
                    rows, imps, preds, tun, evh = _run_security_blockcv(sec, df_raw, scheme, H_run)
                except Exception as e:
                    print(f'  [H={H_run}] {scheme} {sec}: FAILED {repr(e)[:60]}'); continue
                if rows:  _append_csv(pd.DataFrame(rows), blk_csv_p)
                if preds: _append_csv(pd.concat(preds, ignore_index=True), _blk_pred_csv(H_run, OUT_DIR))
                if tun:   _append_csv(pd.DataFrame(tun), _tuning_csv(H_run, OUT_DIR))
                blk_imps += imps
                _dump_pkl(imp_pkl_p, blk_imps)
                _imps_long(blk_imps).to_csv(_blk_imp_csv(H_run, OUT_DIR), index=False)
                print(f'  [H={H_run}] {scheme:<9s} {sec:<10s} ({time.time()-t0:.1f}s)')
        _maybe_push([_blk_csv(H_run, OUT_DIR), _blk_pred_csv(H_run, OUT_DIR), _blk_imp_csv(H_run, OUT_DIR)],
                    f'results: xgb_v5_nor block_cv+loro H{H_run} @ {MACHINE_ID}')
    print('block-CV/LORO sweep done')
else:
    print('[skipped] RUN_BLOCKCV=RUN_LORO=False')

### Assemble metrics + MTC (SEPARATE walk-forward and learnability families)

In [ ]:
def _read_all(globpat):
    import glob
    parts = [pd.read_csv(p) for p in sorted(glob.glob(globpat)) if os.path.getsize(p) > 0]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

wf  = _read_all(_p(OUT_DIR, f'xgb_wf_H*__{MACHINE_ID}.csv'))
blk = _read_all(_p(OUT_DIR, f'xgb_blockcv_H*__{MACHINE_ID}.csv'))
df_all = finalize_long_csv(pd.concat([d for d in (wf, blk) if len(d)], ignore_index=True)) \
         if (len(wf) or len(blk)) else pd.DataFrame(columns=SHARED_COLS)
if len(df_all):
    # Causal forecastability family (walk_forward) — corrected within this estimator
    apply_mtc_family(df_all, ['walk_forward'], 'walk_forward:{horizon×tenor×regime}', model_kind=MODEL_KIND)
    # Non-causal learnability family (block_cv + loro) — kept SEPARATE, never pooled with WF
    apply_mtc_family(df_all, ['block_cv', 'loro'], 'block_cv+loro:{horizon×tenor×regime}', model_kind=MODEL_KIND)
    df_all.to_csv(_p(OUT_DIR, f'xgb_metrics_long__{MACHINE_ID}.csv'), index=False)
    print(f'metrics_long rows={len(df_all)} cols={df_all.shape[1]}')
    print(df_all.groupby('validation_scheme')['security'].count())
else:
    print('no metrics yet (run the sweeps)')

### Figures — importance buckets · SHAP swarm · learning curve

In [ ]:
_imp = _read_all(_p(OUT_DIR, f'xgb_wf_imps_H*__{MACHINE_ID}.csv'))
_eh  = _read_all(_p(OUT_DIR, f'xgb_eval_history_H*__{MACHINE_ID}.csv'))
if not len(_imp):  # fall back to smoke
    _imp = _read_all(_p(SMOKE_DIR, 'xgb_wf_imps_H*.csv'))
    _eh  = _read_all(_p(SMOKE_DIR, 'xgb_eval_history_H*.csv'))

# (a) gain importance by taxonomy bucket
if len(_imp):
    g = _imp[_imp['importance_type'] == 'gain']
    bk = g.groupby('bucket')['value'].sum(); bk = (bk / bk.sum() * 100).sort_values()
    fig, ax = plt.subplots(figsize=(6.4, 3.6))
    ax.barh(bk.index, bk.values, color=OKABE_ITO[2])
    ax.set_xlabel('gain importance share (%)'); ax.set_title('XGBoost gain by feature bucket')
    fig.tight_layout(); fig.savefig(_p(FIG_DIR, 'xgb_importance_buckets.pdf')); plt.close(fig)
    print('saved xgb_importance_buckets.pdf')

# (b) SHAP swarm for a representative fold (recompute on the smoke fold; needs the shap lib for the aesthetic)
try:
    panel = build_panel(df_raw, [SMOKE_SEC], SMOKE_H)
    fc = [c for c in panel.columns if c not in META_COLS]
    ts = pd.Timestamp(WF_FOLDS[-1][1]); te_end = pd.Timestamp(WF_FOLDS[-1][2])
    trn = panel[panel['date'] < ts - pd.tseries.offsets.BDay(SMOKE_H - 1)]
    tst = panel[(panel['date'] >= ts) & (panel['date'] <= te_end)]
    x_tr, x_te, _ = fit_transform_xm_pca(trn[fc], tst[fc])
    x_tr_x, x_te_x = _prepare_x_xgb(x_tr), _prepare_x_xgb(x_te)
    m = xgb.XGBRegressor(n_estimators=200, objective=OBJECTIVE, max_depth=3, learning_rate=0.05,
                         random_state=JULIA_SEED, n_jobs=1, verbosity=0).fit(x_tr_x, trn['y'].to_numpy(float))
    contribs = m.get_booster().predict(xgb.DMatrix(x_te_x, feature_names=list(x_te_x.columns)),
                                       pred_contribs=True)[:, :-1]
    if _HAVE_SHAP:
        shap.summary_plot(contribs, x_te_x, show=False, max_display=20)
        plt.title(f'XGBoost SHAP — {SMOKE_SEC} H={SMOKE_H} (Hiking fold)')
        plt.tight_layout(); plt.savefig(_p(FIG_DIR, 'xgb_shap_swarm.pdf'), bbox_inches='tight'); plt.close()
        print('saved xgb_shap_swarm.pdf')
    else:
        # dependency-free fallback: mean|SHAP| bucket bar
        sa = pd.Series(np.abs(contribs).mean(0), index=list(x_te_x.columns))
        bb = sa.groupby(sa.index.map(bucket_feature)).sum().sort_values()
        fig, ax = plt.subplots(figsize=(6.4, 3.6)); ax.barh(bb.index, bb.values, color=OKABE_ITO[4])
        ax.set_xlabel('mean |SHAP|'); ax.set_title('XGBoost SHAP by bucket (install shap for the swarm)')
        fig.tight_layout(); fig.savefig(_p(FIG_DIR, 'xgb_shap_buckets.pdf')); plt.close(fig)
        print('saved xgb_shap_buckets.pdf (shap lib not installed → bucket bar fallback)')
except Exception as e:
    print(f'[shap fig] skipped — {repr(e)[:80]}')

# (c) learning curve (selected config, representative fold)
if len(_eh):
    one = _eh.sort_values('round')
    key = one[['security', 'horizon', 'fold']].drop_duplicates().iloc[0]
    sub = one[(one.security == key.security) & (one.horizon == key.horizon) & (one.fold == key.fold)]
    fig, ax = plt.subplots(figsize=(6.4, 3.6))
    ax.plot(sub['round'], sub['train_rmse'], label='train', color=OKABE_ITO[0])
    ax.plot(sub['round'], sub['val_rmse'], label='inner-val', color=OKABE_ITO[3])
    ax.set_xlabel('boosting round'); ax.set_ylabel('RMSE')
    ax.set_title(f'XGBoost early-stopping curve — {key.security} H={key.horizon} {key.fold}')
    ax.legend(frameon=False); fig.tight_layout()
    fig.savefig(_p(FIG_DIR, 'xgb_learning_curve.pdf')); plt.close(fig)
    print('saved xgb_learning_curve.pdf')